# Customer Analysis

This is probably the most valuable notebook in the project. Understanding your customers — who's actually driving revenue, how they were acquired, and how long they stick around — is the foundation of any decent retention or growth strategy.

The centrepiece here is RFM segmentation: scoring customers on how recently they bought, how often, and how much they spent. It's a simple framework but it works, and it produces segments that you can actually act on.


In [ ]:

import sys; sys.path.insert(0, '..')
from pipeline.load import get_connection
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

con = get_connection()
c360 = con.execute("SELECT * FROM mart_customer_360").df()
customers = con.execute("SELECT * FROM customers").df()


## RFM segments — who's who

Five segments, from Champions (high recency, high frequency, high spend) down to Lost (haven't bought in ages, low lifetime value). The size of each segment tells you a lot about the health of the customer base.

A healthy business should have a decent Champions cohort and a manageable Lost cohort. If Lost is the biggest segment, you have a retention problem, not an acquisition problem.


In [ ]:

seg = c360[c360['rfm_segment'].notna()]['rfm_segment'].value_counts()
colors = {'Champions':'#2ecc71','Loyal':'#3498db','Potential Loyalist':'#f1c40f','At Risk':'#e67e22','Lost':'#e74c3c'}
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(seg.index, seg.values, color=[colors.get(s, 'gray') for s in seg.index], edgecolor='white')
ax.bar_label(bars, fmt='%d', padding=3)
ax.set_title('RFM Segment Distribution'); ax.set_ylabel('Customers')
plt.xticks(rotation=20); plt.tight_layout(); plt.show()
print(seg.to_frame('count').assign(pct=(seg/seg.sum()*100).round(1)))


## LTV gap between segments

This is the number that should inform where you spend your retention budget. The gap between Champions and At Risk customers shows how much revenue is at stake when a customer slides down the segments. Worth quantifying in dollar terms — "each Champions customer is worth X times more than an At Risk customer" is a much more persuasive argument for retention investment than a bar chart.


In [ ]:

ltv = c360[c360['rfm_segment'].notna()].groupby('rfm_segment')['total_revenue'].mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 4))
ltv.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Average LTV by RFM Segment'); ax.set_ylabel('Avg Revenue ($)')
plt.xticks(rotation=20); plt.tight_layout(); plt.show()
print("LTV gap (Champions vs Lost):", round(ltv['Champions'] / ltv['Lost'], 1), "x")


## Loyalty tier performance

Loyalty tiers are the business's own classification — RFM is our independent calculation. Comparing the two is interesting because it shows whether the loyalty programme is actually correlating with spend behaviour. If Gold customers aren't spending more than Silver, the programme might not be driving the right behaviour.


In [ ]:

tier_rev = c360[c360['has_purchased']].groupby('loyalty_tier')[['total_revenue','total_orders','avg_order_value']].mean().round(2)
print(tier_rev)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, color in zip(axes, tier_rev.columns, ['steelblue','coral','mediumpurple']):
    tier_rev[col].plot(kind='bar', ax=ax, color=color, edgecolor='white')
    ax.set_title(col); ax.set_xlabel('Tier')
    plt.sca(ax); plt.xticks(rotation=0)
plt.suptitle('Loyalty Tier Performance'); plt.tight_layout(); plt.show()


## Acquisition channel — which brings the best customers?

Volume and quality aren't the same thing. A channel that brings in 10,000 customers with low LTV might be less valuable than one bringing in 2,000 high-LTV customers. This breakdown shows you where to invest in acquisition based on actual downstream value — not just sign-up numbers.

Organic typically wins on LTV because people who find you themselves have higher intent. Paid channels often show lower LTV because they capture more passive interest.


In [ ]:

acq = c360[c360['has_purchased']].groupby('acquisition_channel').agg(
    customers=('customer_id', 'count'),
    avg_ltv=('total_revenue', 'mean'),
    avg_orders=('total_orders', 'mean')
).round(2).sort_values('avg_ltv', ascending=False)
print(acq)
fig, ax = plt.subplots(figsize=(9, 4))
acq['avg_ltv'].plot(kind='bar', ax=ax, color='teal', edgecolor='white')
ax.set_title('Avg LTV by Acquisition Channel'); ax.set_ylabel('Avg Revenue ($)')
plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()


## Signup trends over time

Growth in signups is good, but the channel mix matters. If the growth is being driven entirely by Paid Search, that's expensive and fragile. Organic and Referral growth tends to be stickier and cheaper to maintain.


In [ ]:

customers['signup_month'] = pd.to_datetime(customers['signup_date']).dt.to_period('M').astype(str)
monthly_signups = customers.groupby(['signup_month','acquisition_channel'])['customer_id'].count().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(13, 5))
monthly_signups.plot(kind='area', ax=ax, alpha=0.7, stacked=True)
ax.set_title('Monthly Customer Signups by Acquisition Channel')
ax.set_xlabel('Month'); ax.set_ylabel('New Customers')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()


## Revenue by country

Geographic breakdown is useful for two reasons: it tells you where to focus localisation and marketing efforts, and it highlights where you might be leaving money on the table. A country with high customer count but low revenue per customer might have a pricing or product-fit issue.


In [ ]:

country_rev = c360[c360['has_purchased']].groupby('country')['total_revenue'].sum().sort_values(ascending=False).head(10)
fig, ax = plt.subplots(figsize=(9, 4))
country_rev.plot(kind='bar', ax=ax, color='coral', edgecolor='white')
ax.set_title('Top 10 Countries by Total Revenue'); ax.set_ylabel('Revenue ($)')
plt.xticks(rotation=0); plt.tight_layout(); plt.show()
